In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Setup API keys
#########################
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )
# Import tools
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent, LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types


from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext


print("✅ ADK components imported successfully.")

# config retry options
###########################
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

# Agent architecture
###########################

# Flight Agent: with input date and destination, search for the flight options and price
Flight_agent = Agent(
    name="Flight_agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""You are a specialized flight search agent. Your only job is to use the
    google_search tool to find 3 flights with goals as follows based on provided date, starting/ending point.
    1. lowest price.
    2. balance between price and flight time.
    3. shortest flight time.
    You will return these 3 options including the flight number and price.
    """,
    tools=[google_search],
    output_key="flight_options",
)

print("✅ Flight_agent created.")
# Hotel Agent:
# Information agent: generate information better to know before trip.
# Root Agent: Ananlyze trip time, locations, and buget goal, send query to the subagents. Summarize plan and cost. 
root_agent = Agent(
    name="TripCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a trip coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `FlightAgent` tool to find relevant information based on travel time and locations provided by the user.
2. Next, after receiving the flight information, you should choose the proper flight option based on budget provided by the user and list that option out.
""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(Flight_agent)],
)
session_service = InMemorySessionService() #session 
APP_NAME = "default"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session
# Create the Runner
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)


print("✅ root_agent created.")

response = await runner.run_debug(
    "What are the trip plan if I want to travel to yellowstone national park from san jose from 12/02/2025 to 12/23/2025. My budget is $2000."
)